# Final Project Data Mining Minggu 2 E2
Uji efek resolusi native pada YOLO26n. Kedua model dilatih lima epoch dengan batch 4, seed 42, dan 1.244 gambar train dari sumber yang sama. Evaluasi memakai 356 gambar validation dengan 2.205 kotak pada koordinat UTS 640. Notebook menyimpan metrik, prediksi, bobot, log epoch dan benchmark CPU.

In [ ]:
%pip install -q ultralytics==8.4.210 pycocotools==2.0.11

import csv
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
from zipfile import ZipFile
import torch
import ultralytics

cpus = sorted(os.sched_getaffinity(0))[:4]
assert len(cpus) == 4
os.sched_setaffinity(0, cpus)
os.environ.update(OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
torch.set_num_threads(4)
print("Setting up Kaggle CPU training with four threads.", flush=True)
work = Path("/kaggle/working").resolve()
out = work / "E2"
out.mkdir()
source = Path("/tmp/ciringoni-source")
revision = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
subprocess.run(["git", "clone", "-q", "https://github.com/Jordyee/ciringoni-data-mining.git", str(source)], check=True)
subprocess.run(["git", "-C", str(source), "checkout", "-q", "--detach", revision], check=True)
project = source / "Data_Mining_Ciringoni_M1"
scripts = project / "hasil_m2"
anchors = list(Path("/kaggle/input").rglob("audit_pasangan_e2.json"))
assert len(anchors) == 1, f"Harus tersedia satu audit train/val native: {anchors}"
asset = anchors[0].parent
native = Path("/tmp/ciringoni-native")
archive = asset / "native_train_val.zip"
if archive.is_file():
    with ZipFile(archive) as zipped:
        entries = zipped.namelist()
        assert len(entries) == len(set(entries)) == 6400
        assert all(len(name.split("/")) == 4 and ".." not in name.split("/")
                   and name.split("/")[0] in ("640", "1280")
                   and name.split("/")[1] in ("images", "labels")
                   and name.split("/")[2] in ("train", "val") for name in entries)
        zipped.extractall(native)
else:
    extracted = asset / "native_train_val"
    if not extracted.is_dir():
        extracted = asset
    from concurrent.futures import ThreadPoolExecutor
    jobs = []
    for size in ("640","1280"):
        for kind in ("images","labels"):
            for split in ("train","val"):
                key = Path(size,kind,split)
                assert not (extracted / size / kind / "test").exists()
                (native / key).mkdir(parents=True)
                jobs.extend((file, native / key / file.name) for file in (extracted / key).iterdir() if file.is_file())
    assert len(jobs) == 6400
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(lambda job: shutil.copyfile(*job), jobs))
shutil.copy2(asset / "audit_pasangan_e2.json", native)
shutil.copy2(asset / "coco_val_gt.json", project / "hasil_m1")
for size in (640, 1280):
    root = native / str(size)
    data = {"path": str(root), "train": "images/train", "val": "images/val", "names": ["infected"]}
    (root / "data.yaml").write_text(json.dumps(data))
print("Native training and validation are ready.", flush=True)

In [ ]:
sys.path.insert(0, str(scripts))
training_file = scripts / "train_e2_native.py"
training_code = training_file.read_text()
training_code = training_code.replace("    torch.cuda.reset_peak_memory_stats()\n", "")
training_code = training_code.replace("device=0, workers=0", 'device="cpu", workers=0')
training_code = training_code.replace(" | peak_reserved_mib={torch.cuda.max_memory_reserved() / 1024**2:.1f}", "")
training_code = training_code.replace("    torch.cuda.empty_cache()\n", "")
compile(training_code, str(training_file), "exec")
training_file.write_text(training_code)
from train_e2_native import train, verify_pair

verify_pair(native)
runs = out / "runs"
comparisons = [(640, "E2_control_native_640"), (1280, "E2_native_1280")]
weights = asset / "yolo26n.pt"
def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

assert digest(weights) == "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"

for size, name in comparisons:
    train(str(weights), native / str(size) / "data.yaml", runs, name, size, 4, 5)
    with (runs / name / "results.csv").open() as scores:
        assert sum(1 for _ in csv.DictReader(scores)) == 5

In [ ]:
validation = out / "validation"
available = sorted(os.sched_getaffinity(0))
assert len(available) >= 4, f"Empat CPU logis diperlukan untuk benchmark: {available}"
os.sched_setaffinity(0, available[:4])
rows = []
for size, name in comparisons:
    checkpoint = runs / name / "weights/best.pt"
    command = [sys.executable, str(scripts / "evaluasi_e2.py"), str(checkpoint),
               str(native / str(size) / "data.yaml"), str(native / "audit_pasangan_e2.json"),
               str(validation / name), "--size", str(size), "--device", "cpu"]
    subprocess.run(command, check=True)
    subprocess.run(command + ["--cpu-benchmark"], check=True)
    row = json.loads((validation / name / "metrik.json").read_text())
    assert row["gambar"] == 356 and row["split"] == "val"
    row["checkpoint"] = str(checkpoint.relative_to(out))
    row["checkpoint_sha256"] = digest(checkpoint)
    rows.append(row)
with (out / "eksperimen.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

evidence = {"stage": "E2", "training_performed": True, "epochs": 5, "batch": 4, "seed": 42,
            "device": "Kaggle CPU, four threads",
            "python": platform.python_version(), "torch": torch.__version__,
            "ultralytics": ultralytics.__version__, "repo_revision": revision,
            "cpu_affinity": available[:4], "native_audit_sha256": digest(asset / "audit_pasangan_e2.json"),
            "validation_gt_sha256": digest(asset / "coco_val_gt.json"),
            "checkpoints_sha256": {row["id"]: row["checkpoint_sha256"] for row in rows}}
(out / "run.json").write_text(json.dumps(evidence, indent=2))
print((out / "eksperimen.csv").read_text())
print(json.dumps(evidence, indent=2))